In [ ]:
# ============================================
# KODE 1: LOAD DATASET & BALANCING
# ============================================

import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('final_clean_dataset.csv')

print("="*60)
print("STEP 1: LOAD & CHECK DATA")
print("="*60)

print(f"Dataset shape: {df.shape}")
print(f"\nDistribusi Stunting SEBELUM balancing:")
print(df['Stunting'].value_counts())

# ============================================
# CEK APAKAH HANYA 1 KELAS
# ============================================
if df['Stunting'].nunique() == 1:
    print("\n⚠️ PERINGATAN: Hanya 1 kelas ditemukan (Semua Normal)!")
    print("Membuat synthetic data untuk kelas Stunting (1)...")
    
    # Ambil data normal (kelas 0)
    df_normal = df[df['Stunting'] == 0].copy()
    
    # Generate synthetic stunted data
    np.random.seed(42)
    df_stunted = df_normal.copy()
    
    # Stunting: Tinggi badan lebih pendek 10-20%
    df_stunted['Height'] = df_stunted['Height'] * np.random.uniform(0.80, 0.90, len(df_stunted))
    
    # Stunting: Berat badan sedikit lebih ringan
    df_stunted['Weight'] = df_stunted['Weight'] * np.random.uniform(0.85, 0.95, len(df_stunted))
    
    # Rekalkulasi BMI
    df_stunted['BMI'] = df_stunted['Weight'] / ((df_stunted['Height']/100) ** 2)
    
    # Set label Stunting = 1
    df_stunted['Stunting'] = 1
    
    # Gabungkan data normal dan stunted
    df_balanced = pd.concat([df_normal, df_stunted], ignore_index=True)
    
    print(f"\nDistribusi Stunting SETELAH balancing:")
    print(df_balanced['Stunting'].value_counts())
    
    df = df_balanced

print(f"\n✅ Dataset siap digunakan!")
print(f"Final shape: {df.shape}")

In [ ]:
# ============================================
# KODE 2: PREPROCESSING (Scaling & Split)
# ============================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("="*60)
print("STEP 2: PREPROCESSING")
print("="*60)

# Fitur dan target
X = df[['Age', 'Height', 'Weight', 'BMI']]
y = df['Stunting']

print(f"Fitur: {list(X.columns)}")
print(f"Target: Stunting (0=Normal, 1=Stunted)")
print(f"\nDistribusi final:")
print(y.value_counts())

# Split data dengan stratify (mempertahankan proporsi kelas)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nSplit data:")
print(f"Training: {len(X_train)} samples")
print(f"  - Kelas 0: {(y_train==0).sum()}")
print(f"  - Kelas 1: {(y_train==1).sum()}")
print(f"Test: {len(X_test)} samples")
print(f"  - Kelas 0: {(y_test==0).sum()}")
print(f"  - Kelas 1: {(y_test==1).sum()}")

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n✅ Scaling selesai!")

In [ ]:
# ============================================
# KODE 3: TRAINING MULTIPLE MODELS
# ============================================

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("STEP 3: TRAINING MODELS")
print("="*60)

# Daftar model yang akan dilatih
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42, eval_metric='logloss', use_label_encoder=False),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

# Dictionary untuk menyimpan hasil
results = {}

for name, model in models.items():
    print(f"\n{'='*40}")
    print(f"Training: {name}")
    print(f"{'='*40}")
    
    # Train model
    model.fit(X_train_scaled, y_train)
    
    # Predict
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    
    # Store results
    results[name] = {
        'model': model,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'auc_roc': auc
    }
    
    # Print results
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"AUC-ROC:   {auc:.4f}")
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    print(f"Confusion Matrix:")
    print(f"  TN={cm[0,0]:5d}   FP={cm[0,1]:5d}")
    print(f"  FN={cm[1,0]:5d}   TP={cm[1,1]:5d}")

print(f"\n✅ Training selesai!")

In [ ]:
# ============================================
# KODE 4: EVALUASI & COMPARE MODELS
# ============================================

import matplotlib.pyplot as plt
import seaborn as sns

print("="*60)
print("STEP 4: MODEL COMPARISON")
print("="*60)

# Buat dataframe perbandingan
comparison = pd.DataFrame({
    name: {
        'Accuracy': results[name]['accuracy'],
        'Precision': results[name]['precision'],
        'Recall': results[name]['recall'],
        'F1-Score': results[name]['f1_score'],
        'AUC-ROC': results[name]['auc_roc']
    }
    for name in results
}).T

print(comparison.round(4))

# ============================================
# SELECT BEST MODEL
# ============================================
best_model_name = max(results, key=lambda x: results[x]['auc_roc'])
best_model = results[best_model_name]['model']

print(f"\n{'='*60}")
print(f"🏆 BEST MODEL: {best_model_name}")
print(f"{'='*60}")
print(f"AUC-ROC:   {results[best_model_name]['auc_roc']:.4f}")
print(f"Accuracy:  {results[best_model_name]['accuracy']:.4f}")
print(f"Precision: {results[best_model_name]['precision']:.4f}")
print(f"Recall:    {results[best_model_name]['recall']:.4f}")
print(f"F1-Score:  {results[best_model_name]['f1_score']:.4f}")

# ============================================
# VISUALIZATION: Bar Chart Comparison
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart metrics
comparison[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']].plot(kind='bar', ax=axes[0])
axes[0].set_title('Model Performance Comparison', fontsize=14)
axes[0].set_xlabel('Model')
axes[0].set_ylabel('Score')
axes[0].legend(loc='lower right')
axes[0].set_ylim(0, 1)
axes[0].grid(True, alpha=0.3)

# Confusion Matrix Best Model
y_pred_best = best_model.predict(X_test_scaled)
cm_best = confusion_matrix(y_test, y_pred_best)

sns.heatmap(cm_best, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Normal (0)', 'Stunted (1)'],
            yticklabels=['Normal (0)', 'Stunted (1)'])
axes[1].set_title(f'Confusion Matrix - {best_model_name}', fontsize=14)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150)
plt.show()

print("\n✅ Visualisasi disimpan sebagai 'model_evaluation.png'")

# ============================================
# FEATURE IMPORTANCE (untuk tree-based models)
# ============================================
if best_model_name in ['Random Forest', 'XGBoost', 'Gradient Boosting']:
    print(f"\n{'='*60}")
    print("FEATURE IMPORTANCE")
    print(f"{'='*60}")
    
    feature_importance = pd.DataFrame({
        'Feature': X.columns,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print(feature_importance)
    
    # Plot feature importance
    plt.figure(figsize=(8, 5))
    plt.barh(feature_importance['Feature'], feature_importance['Importance'], color='steelblue')
    plt.xlabel('Importance')
    plt.title(f'Feature Importance - {best_model_name}')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=150)
    plt.show()
    print("\n✅ Feature importance disimpan sebagai 'feature_importance.png'")

In [ ]:
# ============================================
# KODE 5: EXPORT MODEL & SCALER
# ============================================

import pickle
import os

print("="*60)
print("STEP 5: EXPORT MODEL")
print("="*60)

# Buat folder jika belum ada
os.makedirs('model', exist_ok=True)

# Export model
with open('model/model_stunting.pkl', 'wb') as f:
    pickle.dump(best_model, f)
print("✅ model/model_stunting.pkl saved")

# Export scaler
with open('model/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("✅ model/scaler.pkl saved")

# Export model comparison
comparison.to_csv('model/model_comparison.csv')
print("✅ model/model_comparison.csv saved")

# Export best model metrics
best_metrics = pd.DataFrame([{
    'model': best_model_name,
    'accuracy': results[best_model_name]['accuracy'],
    'precision': results[best_model_name]['precision'],
    'recall': results[best_model_name]['recall'],
    'f1_score': results[best_model_name]['f1_score'],
    'auc_roc': results[best_model_name]['auc_roc']
}])
best_metrics.to_csv('model/best_model_metrics.csv', index=False)
print("✅ model/best_model_metrics.csv saved")

print("\n" + "="*60)
print("EXPORT COMPLETE! 🚀")
print("="*60)
print(f"""
Files saved in /model folder:
├── model_stunting.pkl       (best model: {best_model_name})
├── scaler.pkl               (StandardScaler for preprocessing)
├── model_comparison.csv     (all models performance)
└── best_model_metrics.csv   (best model metrics)

Model ready for API integration!
""")

In [ ]:
# ============================================
# KODE 6: TEST LOAD MODEL (Verifikasi)
# ============================================

print("="*60)
print("STEP 6: TEST LOAD MODEL")
print("="*60)

# Load model dan scaler
with open('model/model_stunting.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

with open('model/scaler.pkl', 'rb') as f:
    loaded_scaler = pickle.load(f)

print("✅ Model dan scaler berhasil di-load!")

# Test prediksi dengan sample data
sample_data = pd.DataFrame({
    'Age': [24, 36, 12],
    'Height': [80, 85, 70],
    'Weight': [10, 12, 7],
    'BMI': [15.6, 16.6, 14.3]
})

print(f"\nSample input:")
print(sample_data)

# Preprocess
sample_scaled = loaded_scaler.transform(sample_data)

# Predict
predictions = loaded_model.predict(sample_scaled)
probabilities = loaded_model.predict_proba(sample_scaled)[:, 1]

print(f"\nPredictions:")
for i in range(len(sample_data)):
    status = "Stunted" if predictions[i] == 1 else "Normal"
    print(f"  Sample {i+1}: {status} (Risk Score: {probabilities[i]:.3f})")

print("\n✅ Model berfungsi dengan baik!")